# 04. Anomaly Detection & Mismatch Scoring (Isolation Forest)
### Clinical Note-to-Billing Code Consistency Checker (FWA Triage)

In this notebook, we engineer clinical and coding deviation features and train an Isolation Forest to score clinical note-claim pairs for potential billing mismatches.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from clinical_billing_audit.models.anomaly import AnomalyDetector
from clinical_billing_audit.models.ensemble import EnsembleMismatchScorer

## 1. Load Paired Results & Engineer Features

In [ ]:
df = pd.read_csv("../data/processed/paired_with_mismatches.csv")
detector = AnomalyDetector(n_estimators=200, contamination=0.15)
features = detector.engineer_features(df)
print("Engineered Features Shape:", features.shape)
features.head()

## 2. Fit Isolation Forest & Compute Anomaly Scores

In [ ]:
detector.fit(features)
scores = detector.score(features)
flagged_df = detector.flag_mismatches(df, features)

plt.figure(figsize=(10, 5))
sns.histplot(data=flagged_df, x="anomaly_score", hue="is_mismatch", bins=30, kde=True, palette="coolwarm")
plt.title("Isolation Forest Anomaly Score Distribution (Clean vs Injected Mismatch)", fontsize=13)
plt.xlabel("Anomaly Score (Higher = More Suspicious)")
plt.show()